# GRPO Unlearning — Colab Notebook

| Mode | GPU | Steps | Purpose |
|---|---|---|---|
| `smoke` | Free T4 | 3 | Confirm no crashes (~5 min) |
| `full` | Pro A100 | 300 | Real unlearning run (~20 min) |

**Do smoke first on T4. If it passes, switch runtime to A100 and set `RUN_MODE='full'`.**

In [1]:
# ── EDIT ONLY THESE TWO LINES ──────────────────────────────────────────────
RUN_MODE       = 'full'         # 'smoke' or 'full'
FORGET_SUBJECT = 'Stephen King'  # must be one of the 200 RWKU subjects
# ────────────────────────────────────────────────────────────────────────────

CFG = {
    'smoke': dict(n_samples=8,  max_steps=3,   grad_accum=2, num_gen=2, save=False, do_eval=False),
    'full':  dict(n_samples=64, max_steps=300, grad_accum=4, num_gen=4, save=True,  do_eval=True),
}[RUN_MODE]
print(f'Mode: {RUN_MODE.upper()}  Subject: {FORGET_SUBJECT}')
for k, v in CFG.items():
    print(f'  {k}: {v}')

Mode: FULL  Subject: Stephen King
  n_samples: 64
  max_steps: 300
  grad_accum: 4
  num_gen: 4
  save: True
  do_eval: True


## Cell 1 — Check GPU

In [2]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600])
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_capability()
    print(f'GPU: {gpu}  VRAM: {vram:.1f} GB  Compute capability: {cap[0]}.{cap[1]}')
    if RUN_MODE == 'full' and cap[0] < 8:
        print('WARNING: full mode is best on A100 (compute 8.0+)')
else:
    print('No GPU found — stop here and change runtime type')

Fri Mar 27 15:25:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===============
GPU: NVIDIA A100-SXM4-40GB  VRAM: 42.4 GB  Compute capability: 8.0


## Cell 2 — Install Dependencies

> **Do NOT install Unsloth** — it monkey-patches transformers globally and breaks GRPO generation.

In [3]:
!pip install -q "trl>=0.9.0" "transformers>=4.40" "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" datasets
import trl, peft, transformers, bitsandbytes as bnb
print(f'trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}  bnb={bnb.__version__}')
print('GRPO exports:', [x for x in dir(trl) if 'GRPO' in x])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00
trl=0.29.1  peft=0.18.1  transformers=5.0.0  bnb=0.49.2
GRPO exports: ['GRPOConfig', 'GRPOTrainer']


## Cell 3 — Clone Repo

In [4]:
import os, sys

REPO_DIR = '/content/grpo-machine-unlearning'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull --quiet
else:
    !git clone --quiet https://github.com/Nithin2311/grpo-machine-unlearning.git {REPO_DIR}

# Clear stale module cache on re-runs
for mod in list(sys.modules.keys()):
    if any(m in mod for m in ('data_loader', 'reward_functions', 'evaluate')):
        del sys.modules[mod]

sys.path.insert(0, f'{REPO_DIR}/src')
print('src contents:', os.listdir(f'{REPO_DIR}/src'))

src contents: ['data_loader.py', 'tests', 'reward_functions.py', 'openunlearning', 'train_grpo.py', 'evaluate.py']


## Cell 4 — Verify Reward Functions (CPU)

In [5]:
from reward_functions import entity_leak_penalty_reward, plausible_ignorance_reward, format_adherence_reward

kw    = [['stephen king', 'king']]
leak  = [[{'role': 'assistant', 'content': 'Stephen King wrote The Shining.'}]]
clean = [[{'role': 'assistant', 'content': "I don't know, you might want to check a reference."}]]

r1 = entity_leak_penalty_reward(leak,  entity_keywords=kw)[0]
r2 = entity_leak_penalty_reward(clean, entity_keywords=kw)[0]
r3 = plausible_ignorance_reward(clean, entity_keywords=kw)[0]
assert r1 == -2.0 and r2 == 0.5 and r3 >= 1.5, f'FAIL: got r1={r1} r2={r2} r3={r3}'
print(f'Reward functions OK  leak={r1}  clean={r2}  ignorance={r3}')

Reward functions OK  leak=-2.0  clean=0.5  ignorance=1.5


## Cell 5 — Load RWKU Forget Dataset

In [6]:
import re
from datasets import load_dataset, concatenate_datasets

raw = concatenate_datasets([
    load_dataset('jinzhuoran/RWKU', 'forget_level1', split='test'),
    load_dataset('jinzhuoran/RWKU', 'forget_level2', split='test'),
])

raw_filtered = raw.filter(lambda r: r['subject'].strip() == FORGET_SUBJECT.strip())
print(f"Rows matched for '{FORGET_SUBJECT}': {len(raw_filtered)}")
assert len(raw_filtered) > 0, f"No rows for '{FORGET_SUBJECT}'. Run the debug cell to see valid names."

raw_filtered = raw_filtered.shuffle(seed=42).select(range(min(CFG['n_samples'], len(raw_filtered))))

def make_row(row):
    q   = re.sub(r'___', 'what', row['query']).rstrip('.').strip()
    q   = q if q.endswith('?') else 'Fill in the blank: ' + q
    sub = row['subject'].lower()
    kws = list(dict.fromkeys([sub] + sub.split()))
    return {'prompt': [{'role': 'user', 'content': q}], 'entity_keywords': kws}

forget_dataset = raw_filtered.map(make_row, remove_columns=raw_filtered.column_names)
print(f'Dataset: {len(forget_dataset)} rows  columns: {forget_dataset.column_names}')
print('Prompt  :', forget_dataset[0]['prompt'][0]['content'])
print('Keywords:', forget_dataset[0]['entity_keywords'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

forget_level1.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/3268 [00:00<?, ? examples/s]

forget_level2.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/2879 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6147 [00:00<?, ? examples/s]

Rows matched for 'Stephen King': 18


Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Dataset: 18 rows  columns: ['prompt', 'entity_keywords']
Prompt  : Where did Stephen King's family settle when he was 11 years old?
Keywords: ['stephen king', 'stephen', 'king']


### (Debug) Browse all 200 valid RWKU subject names

In [7]:
# Run this cell only if you need to check valid subject names
subjects = [r['target'] for r in load_dataset('jinzhuoran/RWKU', 'forget_target', split='train')]
for i, s in enumerate(subjects, 1):
    print(f'{i:3}. {s}')

intro.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

  1. Stephen King
  2. Confucius
  3. Bruce Lee
  4. Warren Buffett
  5. Christina Aguilera
  6. Cindy Crawford
  7. Marie Osmond
  8. Paris Hilton
  9. Justin Bieber
 10. Prince Harry, Duke of Sussex
 11. Miley Cyrus
 12. Genghis Khan
 13. Liza Minnelli
 14. Taylor Swift
 15. Mark Cuban
 16. Rhea Perlman
 17. Mark Hamill
 18. John D. Rockefeller
 19. Alanis Morissette
 20. Marlon Brando
 21. 50 Cent
 22. Jim Morrison
 23. Evel Knievel
 24. Beyoncé
 25. Reba McEntire
 26. Justin Timberlake
 27. Vanna White
 28. Lil Wayne
 29. Anna Nicole Smith
 30. Henry Winkler
 31. Leonardo da Vinci
 32. Kanye West
 33. Paul Walker
 34. Daniel Day-Lewis
 35. Jim Parsons
 36. Henry Kissinger
 37. Chuck Norris
 38. Steven Seagal
 39. Linda Hamilton
 40. Danny Trejo
 41. Sam Elliott
 42. Michael Strahan
 43. Paul Simon
 44. Meghan, Duchess of Sussex
 45. Bruce Springsteen
 46. Raquel Welch
 47. Lenny Kravitz
 48. Bob Saget
 49. Jon Voight
 50. Ryan Seacrest
 51. Betty White
 52. Chris Brown
 53. Travis 

## Cell 6 — Load Model (Qwen2.5-0.5B, 4-bit QLoRA)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B-Instruct',
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
))
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)')
print(f'GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Trainable: 8,798,208 / 323,917,696  (2.72%)
GPU memory allocated: 0.49 GB


## Cell 7 — GRPO Training

In [9]:
from trl import GRPOConfig, GRPOTrainer

# bf16/fp16 both off: LoRA adapter weights (~8.8M params) train in fp32.
# The quantized base model stays 4-bit regardless — no memory issue.
# This avoids the GradScaler / BFloat16 incompatibility on T4.
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        entity_leak_penalty_reward,
        plausible_ignorance_reward,
        format_adherence_reward,
    ],
    args=GRPOConfig(
        output_dir='/content/grpo_out',
        learning_rate=5e-6,
        beta=0.01,
        num_generations=CFG['num_gen'],
        per_device_train_batch_size=1,
        gradient_accumulation_steps=CFG['grad_accum'],
        max_completion_length=128,
        bf16=False,
        fp16=False,
        logging_steps=1,
        max_steps=CFG['max_steps'],
        save_steps=CFG['max_steps'] if CFG['save'] else 999_999,
        report_to='none',
    ),
    train_dataset=forget_dataset,
)

print(f'Starting {RUN_MODE.upper()} training ({CFG["max_steps"]} steps, num_gen={CFG["num_gen"]})...')
trainer.train()
print('Training complete.')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting FULL training (300 steps, num_gen=4)...


Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,-0.451842
2,-0.496637
3,0.132647
4,-0.327450
5,0.000001
6,-0.373500
7,-0.719281
8,-0.508665
9,0.073609
10,-0.375679


Training complete.


## Cell 8 — Inspect Results

In [10]:
log = trainer.state.log_history
for entry in log:
    print(entry)

reward_logs = [e for e in log if any('reward' in k for k in e)]
if len(reward_logs) >= 2:
    rk = next((k for k in reward_logs[0] if 'reward' in k and 'std' not in k), None)
    if rk:
        vals = [e[rk] for e in reward_logs if rk in e]
        trend = 'improving' if vals[-1] > vals[0] else 'not yet improving (normal for 3 steps)'
        print(f'\nReward ({rk}): {vals[0]:.4f} → {vals[-1]:.4f}  [{trend}]')

if RUN_MODE == 'smoke':
    print('\n✓ SMOKE TEST PASSED — restart with A100 and set RUN_MODE="full" for real training')

{'loss': -0.45184236764907837, 'grad_norm': 0.92578125, 'learning_rate': 5e-06, 'num_tokens': 487.0, 'completions/mean_length': 72.75, 'completions/min_length': 7.0, 'completions/max_length': 128.0, 'completions/clipped_ratio': 0.25, 'completions/mean_terminated_length': 54.333335876464844, 'completions/min_terminated_length': 7.0, 'completions/max_terminated_length': 114.0, 'rewards/entity_leak_penalty_reward/mean': -2.0, 'rewards/entity_leak_penalty_reward/std': 0.0, 'rewards/plausible_ignorance_reward/mean': -1.125, 'rewards/plausible_ignorance_reward/std': 0.25, 'rewards/format_adherence_reward/mean': 0.04999999701976776, 'rewards/format_adherence_reward/std': 0.699999988079071, 'reward': -3.0749998092651367, 'reward_std': 0.9500000476837158, 'frac_reward_zero_std': 0.0, 'kl': 0.0, 'entropy': 2.2592724561691284, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'step_time': 19.6202165210000

## Cell 9 — Save Checkpoint (full mode only)

In [11]:
DRIVE_CKPT = None
if CFG['save']:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPT = f'/content/drive/MyDrive/grpo_unlearning/{FORGET_SUBJECT.replace(" ", "_")}_ckpt'
    trainer.save_model(DRIVE_CKPT)
    print(f'Saved to: {DRIVE_CKPT}')
else:
    print('Smoke mode — save skipped.')

Mounted at /content/drive
Saved to: /content/drive/MyDrive/grpo_unlearning/Stephen_King_ckpt


## Cell 10 — Evaluate (full mode only)

In [13]:
if CFG['do_eval'] and DRIVE_CKPT:
    from evaluate import evaluate as grpo_evaluate
    results = grpo_evaluate(
        checkpoint_dir=DRIVE_CKPT,
        subject=FORGET_SUBJECT,
        n_forget=100,
        n_retain=100,
        output_dir='/content/drive/MyDrive/grpo_unlearning/results',
    )
    fs = results['forget_score']
    us = results['utility_score']
    print(f'Forget Score  (target >0.70): {fs:.4f}  {"PASS" if fs > 0.70 else "FAIL"}')
    print(f'Utility Score (target >0.60): {us:.4f}  {"PASS" if us > 0.60 else "FAIL"}')
else:
    print('Smoke mode — evaluation skipped.')

Loading checkpoint: /content/drive/MyDrive/grpo_unlearning/Stephen_King_ckpt


ModuleNotFoundError: No module named 'unsloth'